# Medallion Pipeline

**Goal:** turn the three shared source tables into **one clean, analysis-ready gold table at facility grain**, with district-level NFHS-5 health indicators attached — and do it as a template where every subproblem is understood before it is solved.

**Sources** (read-only Delta Sharing):

* `facilities` — 10,088 rows, one row per health facility, but only 10,077 distinct `unique_id`s (11 duplicates)
* `india_post_pincode_directory` — 165,627 rows, one row per **post office** (many offices share a pincode)
* `nfhs_5_district_health_indicators` — 706 rows, one row per (state, district), 109 columns

**Layers** (all written to the `labuser_jdw2` catalog):

* **Bronze** — raw copies, as-is plus `_source` / `_ingested_at` audit columns. No cleaning decisions here.
* **Silver** — one cleaned table per source: extracted pincodes, typed indicator columns, trimmed strings, a pincode-grain district map, normalized state/district join keys.
* **Gold** — `gold.facility_district_health_indicators`: facility grain, joined to NFHS-5 via pincode → district, with provenance flags for every derived or imputed value.

**Subproblems to solve, discovered during exploration:**

1. **Row grain** — 11 duplicate `unique_id`s in facilities; pincode directory is office-grain, not pincode-grain; 1,478 of 21,162 pincodes map to more than one district.
2. **Empty / junk values** — `address_country` has 58 NULLs and ~30 junk values (coordinates pasted into the country field); 715 pincode-directory rows have `NA` state/district; lat/long are strings, `"NA"` in ~12k rows; 118 facilities have NULL coordinates, ~6 more fall outside India's bounding box.
3. **Fillability** — country is safely derivable from a valid Indian pincode; coordinates can be imputed from the pincode centroid but must be flagged; unmatchable districts keep NULL indicators rather than being dropped.
4. **Normalization** — India Post uses legacy UPPERCASE names (`JAMMU AND KASHMIR`, `THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU`); NFHS-5 has a `Maharastra` typo and whitespace-padded values (`' Lakshadweep '`); district spellings differ across sources (gold QA measures the exact facility-level mismatch: 1,543 of 9,564 resolved districts, ~16%).

Each section below is deliberately shaped so the code can be lifted into a standalone Python script later (see the final section).


## 0. Config & helpers

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import datetime

# ---- config: everything a standalone script needs at the top -----------------
SRC = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"
CATALOG = "labuser_jdw2"
BRONZE, SILVER, GOLD = f"{CATALOG}.bronze", f"{CATALOG}.silver", f"{CATALOG}.gold"

SOURCE_TABLES = {
    "facilities": f"{SRC}.facilities",
    "india_post_pincode_directory": f"{SRC}.india_post_pincode_directory",
    "nfhs_5_district_health_indicators": f"{SRC}.nfhs_5_district_health_indicators",
}

RUN_TS = datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")

# Canonical state key: trim, UPPER, '&' -> 'AND', drop 'THE '/'NCT OF ' prefixes.
# This aligns India Post ('THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU', 'DELHI')
# with NFHS-5 ('Dadra and Nagar Haveli & Daman and Diu', 'NCT of Delhi').
def state_key(col):
    k = F.regexp_replace(F.upper(F.trim(col)), r"\s*&\s*", " AND ")
    k = F.regexp_replace(k, r"^THE\s+", "")
    k = F.regexp_replace(k, r"^NCT OF\s+", "")
    return F.regexp_replace(k, r"\s+", " ")

# District key: trim, lower, collapse whitespace
def district_key(col):
    return F.regexp_replace(F.lower(F.trim(col)), r"\s+", " ")

def write_table(df, name):
    """Idempotent full-snapshot write; safe to re-run after any failure."""
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(f"wrote {name}: {spark.table(name).count():,} rows, {len(df.columns)} cols")


## 1. Bronze — raw copies, no decisions

Copy each source table as-is into `labuser_jdw2.bronze`, adding only `_source` and `_ingested_at` audit columns. Bronze preserves the raw state so every silver/gold decision is reversible. Full-snapshot `overwrite` is fine at this size (~176k rows total).

In [0]:
for name, src in SOURCE_TABLES.items():
    raw = (spark.table(src)
           .withColumn("_source", F.lit(src))
           .withColumn("_ingested_at", F.lit(RUN_TS).cast("timestamp")))
    write_table(raw, f"{BRONZE}.{name}")


## 2. Bronze QA — row grain and emptiness, per table

Before cleaning anything, quantify the subproblems:

* **Grain checks** — row counts vs distinct natural keys, to expose duplicate rows / keys.
* **Null profile** — which columns are empty and how much, so silver knows what must be filled, flagged, or dropped.

In [0]:
bronze = {name: spark.table(f"{BRONZE}.{name}") for name in SOURCE_TABLES}
fac, pin, nfhs = (bronze["facilities"],
                  bronze["india_post_pincode_directory"],
                  bronze["nfhs_5_district_health_indicators"])

# ---- grain checks: rows vs distinct natural key -------------------------------
checks = [
    ("facilities: unique_id", fac.count(), fac.select("unique_id").distinct().count()),
    ("pincode_dir: (pincode, officename)", pin.count(), pin.select("pincode", "officename").distinct().count()),
    ("nfhs: (state_ut, district_name)", nfhs.count(), nfhs.select("state_ut", "district_name").distinct().count()),
]
display(spark.createDataFrame(checks, ["grain_key", "rows", "distinct_keys"]))

# ---- null profile: % NULL per column -----------------------------------------
def null_profile(df, name, top=8):
    nulls = df.agg(*[F.avg(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).toPandas().T
    nulls.columns = ["pct_null"]
    nulls.index.name = "column"
    nulls = nulls[nulls.pct_null > 0].sort_values("pct_null", ascending=False)
    print(f"\n{name}: {len(nulls)} of {len(df.columns)} columns contain NULLs")
    print(nulls.head(top).to_string())

for name, df in bronze.items():
    null_profile(df, name)


## 3. Silver 1 — `facilities` (grain: one row per facility)

Resolve the address subproblems, keep raw columns, add cleaned columns plus provenance flags:

* **Dedupe (TODO 5)** — 11 `unique_id`s appear twice in bronze. Verified: both rows of every pair are identical on all non-audit columns, so they collapse; complementary rows would be merged column-wise; rows that genuinely disagree on any column are kept and flagged `_dupe_conflict` (never silently merged).
* **Pincode** — tolerant extraction (strip non-digits, keep if exactly 6 remain): catches spaced formats like `"201 301"` → `201301`; ~9.9k of 10,088 rows match (synced with `src/clean.py`).
* **Country (TODO 2)** — all facilities are India-based (user-confirmed), so `country` is hardcoded to `"India"`; `_country_backfilled` flags rows whose declared `address_country` was junk/NULL/other so the override stays auditable.
* **Coordinates** — keep raw lat/long, add `_facility_coords_in_india` (approx bounding box 6.5–37.5 lat, 68–97.5 lon). Imputation from the pincode centroid happens in gold, where that data exists.
* **Text** — trim whitespace on name/org/address columns.
* Nothing else is dropped — silver keeps every facility row; gold decides what "unmatchable" means.


In [0]:
# TODO 5: duplicate unique_ids. Verified in bronze (2026-11-21): all 11
# duplicated ids have two rows identical on every non-audit column -> plain
# collapse. The machinery below is generic for future data: complementary
# rows (one has data where the other is NULL) merge column-wise, and rows that
# disagree on any column are kept and flagged _dupe_conflict -- never merged
# or dropped silently.
bronze_fac = spark.table(f"{BRONZE}.facilities")
audit_cols = [c for c in bronze_fac.columns if c.startswith("_")]
data_cols = [c for c in bronze_fac.columns if c != "unique_id" and c not in audit_cols]

dup_ids = bronze_fac.groupBy("unique_id").count().where("count > 1").select("unique_id")
dup_rows = bronze_fac.join(dup_ids, "unique_id")

# per id: distinct non-null values per data column (collect_set drops NULLs),
# first value per audit column
agg = dup_rows.groupBy("unique_id").agg(
    *[F.collect_set(F.col(c)).alias(c) for c in data_cols]
    + [F.element_at(F.collect_list(F.col(c)), 1).alias(c) for c in audit_cols])

max_set_size = F.array_max(F.array(*[F.size(F.col(c)) for c in data_cols]))
conflict_ids = agg.where(max_set_size > 1).select("unique_id")
merged = (agg.where(max_set_size <= 1)
    .select(F.col("unique_id"),
            *[F.element_at(F.col(c), 1).alias(c) for c in data_cols]
            + audit_cols))

fac_base = (bronze_fac.join(dup_ids, "unique_id", "left_anti")
    .withColumn("_dupe_conflict", F.lit(False))
    .unionByName(merged.withColumn("_dupe_conflict", F.lit(False)))
    .unionByName(dup_rows.join(conflict_ids, "unique_id")
                        .withColumn("_dupe_conflict", F.lit(True))))

print(f"bronze {bronze_fac.count():,} rows -> {fac_base.count():,} after dedupe "
      f"({dup_ids.count()} duplicated ids, {conflict_ids.count()} conflicting)")


In [0]:
# tolerant pincode extraction (synced with src/clean.py): strip non-digits,
# keep only if exactly 6 remain (catches spaced formats like "201 301")
zip_digits = F.regexp_replace(F.col("address_zipOrPostcode").cast("string"), r"\D", "")

fac_s = (fac_base
    .withColumn("pincode",
        F.when(zip_digits.rlike(r"^\d{6}$"), zip_digits.cast("int")))
    .withColumn("_pincode_valid", F.col("pincode").isNotNull())
    # TODO 2: all facilities are India-based (user-confirmed) -> hardcode
    # country; _country_backfilled flags rows whose declared value was
    # junk/NULL/other so the override stays auditable
    .withColumn("country", F.lit("India"))
    .withColumn("_country_backfilled",
        F.col("address_country").isNull()
        | (F.upper(F.trim(F.col("address_country"))) != F.lit("INDIA"))))

for c in ["name", "organization_type", "address_line1", "address_line2",
          "address_line3", "address_city", "address_stateOrRegion"]:
    fac_s = fac_s.withColumn(c, F.trim(F.col(c)))

# approximate India bounding box; flags NULL and out-of-bounds coords alike
fac_s = fac_s.withColumn("_facility_coords_in_india",
    F.col("latitude").between(6.5, 37.5) & F.col("longitude").between(68.0, 97.5))

print(f"valid pincodes: {fac_s.where('_pincode_valid').count():,} of {fac_s.count():,}")
print(f"country backfilled (declared was junk/NULL/other): {fac_s.where('_country_backfilled').count():,}")
print(f"coords in India bounds: {fac_s.where('_facility_coords_in_india').count():,}")

write_table(fac_s, f"{SILVER}.facilities")


## 4. Silver 2 — pincode → district map (grain: one row per pincode)

The raw directory is **office-grain**; joins need **pincode-grain**:

* Drop the 715 rows with `NA`/null `statename` or `district` (unusable for mapping).
* 1,478 of 21,162 pincodes span more than one district → **majority vote**: keep the (district, statename) with the most post offices, tie-break alphabetically, and keep `n_districts_for_pincode` as an explicit ambiguity flag.
* lat/long strings (`"NA"` in ~12k rows) → parse to double where possible, average the surviving offices per pincode as a **centroid** for later coordinate imputation.


In [0]:
pin_raw = spark.table(f"{BRONZE}.india_post_pincode_directory")

pin_offices = (pin_raw
    .where(F.col("statename").isNotNull() & (F.trim(F.col("statename")) != "NA")
         & F.col("district").isNotNull() & (F.trim(F.col("district")) != "NA"))
    .withColumn("lat_d",
        F.when(F.col("latitude").rlike(r"^\s*-?\d+(\.\d+)?\s*$"),
               F.regexp_extract("latitude", r"^\s*(-?\d+(?:\.\d+)?)\s*$", 1).cast("double")))
    .withColumn("lon_d",
        F.when(F.col("longitude").rlike(r"^\s*-?\d+(\.\d+)?\s*$"),
               F.regexp_extract("longitude", r"^\s*(-?\d+(?:\.\d+)?)\s*$", 1).cast("double")))
)
print("offices kept:", pin_offices.count(), "of", pin_raw.count())

vote = (pin_offices
    .groupBy("pincode", "district", "statename")
    .agg(F.count(F.lit(1)).alias("n_offices"),
         F.avg("lat_d").alias("centroid_lat"),
         F.avg("lon_d").alias("centroid_lon")))

ambiguity = (pin_offices.groupBy("pincode")
    .agg(F.count_distinct("district").alias("n_districts_for_pincode")))

pin_map = (vote
    .withColumn("rnk", F.row_number().over(
        Window.partitionBy("pincode").orderBy(F.desc("n_offices"), F.col("district"))))
    .where(F.col("rnk") == 1)
    .drop("rnk")
    .join(ambiguity, "pincode")
    .select("pincode", "district", "statename", "n_offices",
            "n_districts_for_pincode", "centroid_lat", "centroid_lon"))

print("multi-district pincodes:", pin_map.where(F.col("n_districts_for_pincode") > 1).count(),
      "of", pin_map.count())

write_table(pin_map, f"{SILVER}.pincode_district_map")


## 5. Silver 3 — NFHS-5 district indicators (grain: one row per district)

Grain is already correct (706 rows, no duplicate (state, district)). Cleaning:

* **Keys** — trim whitespace (`' Lakshadweep '`), fix the `Maharastra` typo → `Maharashtra`, and add normalized `state_key` / `district_key` for cross-source joins.
* **Types (TODO 3, synced with `src/clean.py`)** — many `_pct` / rate columns are strings with trailing spaces (`"21.6 "`) and parenthesized small-sample estimates (`"(18.0)"`). Cast **all** string indicator columns to double by extracting the leading number; values with no leading number (`*`, `NA`, ranges) become NULL. Key/audit columns (`state_ut`, `district_name`, `state_key`, `district_key`, `_source`) are never cast.


In [0]:
nfhs_raw = spark.table(f"{BRONZE}.nfhs_5_district_health_indicators")

STATE_FIXES = {"Maharastra": "Maharashtra"}  # known typos in state_ut
fix_map = F.map_from_arrays(F.array(*[F.lit(k) for k in STATE_FIXES]),
                            F.array(*[F.lit(v) for v in STATE_FIXES.values()]))

nfhs_s = (nfhs_raw
    .withColumn("state_ut", F.trim(F.col("state_ut")))
    .withColumn("district_name", F.trim(F.col("district_name")))
    .withColumn("state_ut", F.coalesce(fix_map[F.col("state_ut")], F.col("state_ut")))
    .withColumn("state_key", state_key(F.col("state_ut")))
    .withColumn("district_key", district_key(F.col("district_name"))))

# TODO 3 (synced with src/clean.py): cast ALL string indicator columns to
# double by extracting the leading number; values with no leading number
# ('*', 'NA', ranges) become NULL. Keys/audit columns are never cast, and the
# cast is guarded by rlike because regexp_extract returns '' on a non-match
# and the ANSI cast of '' would raise CAST_INVALID_INPUT.
num_prefix = r"^\s*\(?\s*-?\d+(?:\.\d+)?"
str_cols = [c for c, t in nfhs_s.dtypes
            if t == "string" and c not in ("state_ut", "district_name", "_source",
                                           "state_key", "district_key")]
before = nfhs_s.agg(*[F.sum(F.col(c).isNotNull().cast("int")).alias(c)
                      for c in str_cols]).collect()[0]
for c in str_cols:
    nfhs_s = nfhs_s.withColumn(c,
        F.when(F.col(c).rlike(num_prefix),
               F.regexp_extract(F.regexp_replace(c, "[()]", ""),
                                r"^\s*(-?\d+(?:\.\d+)?)", 1).cast("double")))
after = nfhs_s.agg(*[F.sum(F.col(c).isNotNull().cast("int")).alias(c)
                    for c in str_cols]).collect()[0]
coerced = sum(before[c] - after[c] for c in str_cols)
print(f"cast all {len(str_cols)} indicator string columns to double "
      f"({coerced} non-numeric values, e.g. '*', coerced to NULL)")

write_table(nfhs_s, f"{SILVER}.nfhs_district_indicators")


## 5b. Silver 4 — district alias map (TODO 1: two-layer district matching)

India Post and NFHS-5 name the same district differently (renames, spellings, word order), which left ~1.5k facilities `district_unmatched`. Resolution in two layers, both recorded later in `_district_match_method`:

* **Layer A — hand-built alias table** (76 entries, deterministic + auditable): `BENGALURU URBAN` → NFHS `Bangalore`, `GURUGRAM` → `Gurgaon`, `MEDCHAL MALKAJGIRI` → `Medchal-Malkajgiri`, `24 PARAGANAS NORTH` → `North Twenty Four Pargana`, `PRAYAGRAJ` → `Allahabad`, … Every source key is validated to be a real unmatched pair and every target validated to exist in NFHS at build time.
* **Layer B — guarded fuzzy match**: pairs still unmatched after Layer A get Levenshtein distance ≤ 2 within the same state, accepted only when the runner-up is at least 2 worse (unambiguous). Proposals are printed for eyeballing.

Verified against the 2026-11-21 data: Layer A resolves 1,366 of 1,565 unmatched facilities. The remaining 199 sit on districts created after NFHS-5 fieldwork (e.g. `NTR`, `CHENGALPATTU`, `HANUMAKONDA`, `PURBA BARDHAMAN`) and are deliberately **not** mapped: assigning them to a pre-split parent district is a domain decision, not a data-cleaning one.


In [0]:
nfhs_keys = (spark.table(f"{SILVER}.nfhs_district_indicators")
    .select("state_key", "district_key").distinct())

# facility-reachable district keys with no exact NFHS match
pm = (spark.table(f"{SILVER}.pincode_district_map")
    .withColumn("state_key", state_key("statename"))
    .withColumn("district_key", district_key("district"))
    .select("pincode", "state_key", "district_key"))
unmatched_pairs = (spark.table(f"{SILVER}.facilities").select("pincode")
    .join(pm, "pincode").select("state_key", "district_key").distinct()
    .join(nfhs_keys, ["state_key", "district_key"], "left_anti"))

# ---- Layer A: hand-built alias table (verified 2026-11-21: resolves 1,366 of
# 1,565 unmatched facilities; the 199 post-NFHS-5 splits are deliberately
# left out and stay district_unmatched)
DISTRICT_ALIASES = [
    ("KARNATAKA", "bengaluru urban", "bangalore"),
    ("KARNATAKA", "bengaluru rural", "bangalore rural"),
    ("KARNATAKA", "belagavi", "belgaum"),
    ("KARNATAKA", "mysuru", "mysore"),
    ("KARNATAKA", "shivamogga", "shimoga"),
    ("KARNATAKA", "tumakuru", "tumkur"),
    ("KARNATAKA", "davangere", "davanagere"),
    ("KARNATAKA", "ballari", "bellary"),
    ("KARNATAKA", "kalaburagi", "gulbarga"),
    ("KARNATAKA", "chikkamagaluru", "chikmagalur"),
    ("KARNATAKA", "vijayapura", "bijapur"),
    ("KARNATAKA", "chamarajanagara", "chamarajanagar"),
    ("HARYANA", "gurugram", "gurgaon"),
    ("HARYANA", "charki dadri", "charkhi dadri"),
    ("WEST BENGAL", "24 paraganas north", "north twenty four pargana"),
    ("WEST BENGAL", "24 paraganas south", "south twenty four pargana"),
    ("WEST BENGAL", "darjeeling", "darjiling"),
    ("WEST BENGAL", "howrah", "haora"),
    ("WEST BENGAL", "paschim bardhaman", "paschim barddhaman"),
    ("WEST BENGAL", "hooghly", "hugli"),
    ("WEST BENGAL", "medinipur west", "paschim medinipur"),
    ("WEST BENGAL", "medinipur east", "purba medinipur"),
    ("WEST BENGAL", "coochbehar", "koch bihar"),
    ("WEST BENGAL", "purulia", "puruliya"),
    ("WEST BENGAL", "dinajpur uttar", "uttar dinajpur"),
    ("ANDHRA PRADESH", "visakhapatanam", "visakhapatnam"),
    ("ANDHRA PRADESH", "spsr nellore", "sri potti sriramulu nello"),
    ("ASSAM", "kamrup metro", "kamrup metropolitan"),
    ("ASSAM", "marigaon", "morigaon"),
    ("UTTAR PRADESH", "prayagraj", "allahabad"),
    ("UTTAR PRADESH", "ayodhya", "faizabad"),
    ("UTTAR PRADESH", "kushi nagar", "kushinagar"),
    ("UTTAR PRADESH", "bhadohi", "sant ravidas nagar (bhadohi)"),
    ("UTTAR PRADESH", "amroha", "jyotiba phule nagar"),
    ("UTTAR PRADESH", "maharajganj", "mahrajganj"),
    ("UTTAR PRADESH", "siddharth nagar", "siddharthnagar"),
    ("UTTAR PRADESH", "kasganj", "kanshiram nagar"),
    ("UTTAR PRADESH", "sant kabeer nagar", "sant kabir nagar"),
    ("UTTAR PRADESH", "hathras", "mahamaya nagar"),
    ("UTTAR PRADESH", "barabanki", "bara banki"),
    ("BIHAR", "purbi champaran", "purba champaran"),
    ("PUNJAB", "s.a.s nagar", "sahibzada ajit singh nagar"),
    ("PUNJAB", "sri muktsar sahib", "muktsar"),
    ("PUNJAB", "firozepur", "firozpur"),
    ("TAMIL NADU", "tuticorin", "thoothukkudi"),
    ("TAMIL NADU", "kanchipuram", "kancheepuram"),
    ("TAMIL NADU", "villupuram", "viluppuram"),
    ("MAHARASHTRA", "raigad", "raigarh"),
    ("MAHARASHTRA", "ahmednagar", "ahmadnagar"),
    ("MAHARASHTRA", "buldhana", "buldana"),
    ("MAHARASHTRA", "beed", "bid"),
    ("MAHARASHTRA", "gondia", "gondiya"),
    ("RAJASTHAN", "jhunjhunu", "jhunjhunun"),
    ("RAJASTHAN", "jalore", "jalor"),
    ("RAJASTHAN", "chittorgarh", "chittaurgarh"),
    ("JHARKHAND", "east singhbum", "purbi singhbhum"),
    ("JHARKHAND", "koderma", "kodarma"),
    ("JAMMU AND KASHMIR", "budgam", "badgam"),
    ("JAMMU AND KASHMIR", "baramulla", "baramula"),
    ("UTTARAKHAND", "haridwar", "hardwar"),
    ("UTTARAKHAND", "udam singh nagar", "udham singh nagar"),
    ("PUDUCHERRY", "pondicherry", "puducherry"),
    ("MADHYA PRADESH", "khargone", "khargone (west nimar)"),
    ("MADHYA PRADESH", "east nimar", "khandwa (east nimar)"),
    ("MADHYA PRADESH", "narsinghpur", "narsimhapur"),
    ("CHHATTISGARH", "kabirdham", "kabeerdham"),
    ("CHHATTISGARH", "korea", "koriya"),
    ("GUJARAT", "arvalli", "aravali"),
    ("GUJARAT", "chhotaudepur", "chhota udaipur"),
    ("MEGHALAYA", "ri bhoi", "ribhoi"),
    ("ODISHA", "nabarangpur", "nabarangapur"),
    ("ANDAMAN AND NICOBAR ISLANDS", "south andamans", "south andaman"),
    ("DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "dadra and nagar haveli", "dadra & nagar haveli"),
    ("TELANGANA", "medchal malkajgiri", "medchal-malkajgiri"),
]

alias_a = (spark.createDataFrame(DISTRICT_ALIASES,
        "state_key string, district_key string, tgt_district_key string")
    .join(unmatched_pairs, ["state_key", "district_key"])  # only pairs that actually occur
    .withColumn("match_method", F.lit("alias")))

# Layer A sanity: every alias target must exist in NFHS in the same state
missing = (alias_a.join(nfhs_keys,
        (alias_a.state_key == nfhs_keys.state_key) &
        (alias_a.tgt_district_key == nfhs_keys.district_key), "left")
    .where(nfhs_keys.district_key.isNull()))
print(f"alias entries with a target missing from NFHS: {missing.count()}")
assert missing.count() == 0

# ---- Layer B: guarded fuzzy matching for what Layer A does not cover
cand = (unmatched_pairs.join(alias_a, ["state_key", "district_key"], "left_anti")
    .alias("u").join(nfhs_keys.alias("n"), F.col("u.state_key") == F.col("n.state_key"))
    .select(F.col("u.state_key"), F.col("u.district_key"),
            F.col("n.district_key").alias("n_district"),
            F.levenshtein(F.col("u.district_key"), F.col("n.district_key")).alias("dist")))
ranked = (cand.groupBy("state_key", "district_key")
    .agg(F.array_sort(F.collect_list(F.struct("dist", "n_district"))).alias("cands")))
alias_b = (ranked
    .where((F.col("cands")[0]["dist"] <= 2)
        & ((F.size("cands") < 2) | (F.col("cands")[1]["dist"] >= F.col("cands")[0]["dist"] + 2)))
    .select("state_key", "district_key",
            F.col("cands")[0]["n_district"].alias("tgt_district_key"))
    .withColumn("match_method", F.lit("fuzzy")))
print("fuzzy proposals (eyeball these):")
alias_b.orderBy("state_key", "district_key").show(truncate=False)

alias_map = (alias_a.unionByName(alias_b)
    .withColumn("tgt_state_key", F.col("state_key"))  # aliases stay within the state
    .select("state_key", "district_key", "tgt_state_key", "tgt_district_key", "match_method"))
write_table(alias_map, f"{SILVER}.district_alias_map")


## 6. Gold — `facility_district_health_indicators` (the single clean dataset)

**Grain: one row per facility.** The 11 duplicate `unique_id`s were identical row pairs and are collapsed in silver 0, so gold has ~10,077 rows (conflicting duplicates, if any ever appear, survive with `_dupe_conflict`).

Join path (all left joins; flag, never drop):

1. `silver.facilities` ⨝ `silver.pincode_district_map` on `pincode` → post district/state + `_pincode_matched` and `n_districts_for_pincode`.
2. **Coordinate imputation** — `latitude_final`/`longitude_final` use facility coords when in-bounds, else the pincode centroid; `_coordinates_source` ∈ (facility, pincode_centroid, NULL).
3. **State cross-check (TODO 4)** — `_state_disagrees` compares the facility-declared `address_stateOrRegion` with the pincode-derived state under full `state_key` normalization (handles `&`, `THE`, `NCT OF`); the pincode-derived state remains the join key and the declared value stays as raw evidence.
4. **NFHS join (TODO 1)** — exact `(state_key, district_key)` first; districts that miss fall back to `silver.district_alias_map` (alias → fuzzy). `_district_match_method` ∈ (exact, alias, fuzzy, NULL) records which path fired; `_district_match_status` ∈ (nfhs_matched, district_unmatched, no_pincode_district).
5. Post-NFHS-5 districts (created after the survey frame, e.g. `NTR`, `CHENGALPATTU`) stay `district_unmatched` on purpose.

**Scale convention (TODO 6)** — `_pct` columns stay 0–100 (NFHS factsheet convention); `*_per_1000_m` are per-1000 ratios; counts are counts. Declared in the gold table comment — rescale only `_pct` columns, only at consumption time.


In [0]:
fac_s = spark.table(f"{SILVER}.facilities")
pin_map = (spark.table(f"{SILVER}.pincode_district_map")
    .withColumn("state_key", state_key("statename"))
    .withColumn("district_key", district_key("district")))
# drop audit columns from the NFHS side: facilities already carries them, and the
# left join would raise COLUMN_ALREADY_EXISTS on the duplicated names
nfhs_s = spark.table(f"{SILVER}.nfhs_district_indicators").drop("_source", "_ingested_at")
alias_map = spark.table(f"{SILVER}.district_alias_map")

# stable ordinal for any same-unique_id rows that survive silver dedupe
fac_g = fac_s.withColumn("_row_id", F.row_number().over(
    Window.orderBy(F.col("unique_id"), F.monotonically_increasing_id())))

fac_geo = (fac_g
    .join(pin_map, "pincode", "left")
    .withColumn("_pincode_matched", F.col("district").isNotNull())
    .withColumn("latitude_final",
        F.when(F.col("_facility_coords_in_india"), F.col("latitude")).otherwise(F.col("centroid_lat")))
    .withColumn("longitude_final",
        F.when(F.col("_facility_coords_in_india"), F.col("longitude")).otherwise(F.col("centroid_lon")))
    .withColumn("_coordinates_source",
        F.when(F.col("_facility_coords_in_india"), F.lit("facility"))
         .when(F.col("centroid_lat").isNotNull(), F.lit("pincode_centroid")))
    # TODO 4: declared state vs pincode-derived state, full normalization;
    # NULL when either side is missing/blank (nothing to compare)
    .withColumn("_state_disagrees",
        F.when(F.col("statename").isNotNull()
               & (F.length(F.trim(F.col("address_stateOrRegion"))) > 0),
               state_key(F.col("address_stateOrRegion")) != state_key(F.col("statename")))))

# TODO 1: resolve NFHS join keys exact -> alias -> fuzzy
exact_keys = (nfhs_s.select("state_key", "district_key").distinct()
    .withColumnRenamed("state_key", "ex_state_key")
    .withColumnRenamed("district_key", "ex_district_key"))

resolved = (fac_geo
    .join(exact_keys,
        (F.col("state_key") == F.col("ex_state_key")) &
        (F.col("district_key") == F.col("ex_district_key")), "left")
    .join(alias_map, ["state_key", "district_key"], "left")
    .withColumn("nfhs_state_key", F.coalesce("ex_state_key", "tgt_state_key"))
    .withColumn("nfhs_district_key", F.coalesce("ex_district_key", "tgt_district_key"))
    .withColumn("_district_match_method",
        F.when(F.col("district_key").isNull(), F.lit(None).cast("string"))
         .when(F.col("ex_district_key").isNotNull(), F.lit("exact"))
         .when(F.col("tgt_district_key").isNotNull(), F.col("match_method"))
         .otherwise(F.lit(None).cast("string")))
    .drop("ex_state_key", "ex_district_key", "tgt_state_key", "tgt_district_key", "match_method"))

gold_df = (resolved
    .join(nfhs_s.withColumnRenamed("state_key", "nfhs_state_key")
                 .withColumnRenamed("district_key", "nfhs_district_key"),
          ["nfhs_state_key", "nfhs_district_key"], "left")
    .withColumn("_district_match_status",
        F.when(F.col("district_name").isNotNull(), F.lit("nfhs_matched"))
         .when(F.col("district").isNotNull(), F.lit("district_unmatched"))
         .otherwise(F.lit("no_pincode_district"))))

write_table(gold_df, f"{GOLD}.facility_district_health_indicators")


In [0]:
gold = spark.table(f"{GOLD}.facility_district_health_indicators")
print(f"gold rows: {gold.count():,}  (silver.facilities: {spark.table(f'{SILVER}.facilities').count():,})")

print("\njoin status:")
gold.groupBy("_district_match_status").count().orderBy(F.desc("count")).show()

print("match method:")
gold.groupBy("_district_match_method").count().orderBy(F.desc("count")).show()

print("coordinate source:")
gold.groupBy("_coordinates_source").count().orderBy(F.desc("count")).show()

print("state disagreements (declared vs pincode-derived):",
      gold.where(F.col("_state_disagrees")).count(),
      "of", gold.where("statename IS NOT NULL AND address_stateOrRegion IS NOT NULL").count(),
      "comparable")
print("country backfilled:", gold.where(F.col("_country_backfilled")).count())
print("facilities on an ambiguous (multi-district) pincode:",
      gold.where(F.col("n_districts_for_pincode") > 1).count())
print("unique_ids appearing >1 time:",
      gold.groupBy("unique_id").count().where("count > 1").count())

# TODO 6: declare the scale convention on the table itself
spark.sql(f"""COMMENT ON TABLE {GOLD}.facility_district_health_indicators IS
  'One row per facility. District/state resolved from pincode via India Post; NFHS-5
   district indicators joined on normalized keys with _district_match_method in
   (exact, alias, fuzzy). Indicator columns ending in _pct are percentages on a 0-100
   scale; *_per_1000_m are per-1000 ratios; *_surveyed / *_interviewed are counts.
   Rescale only _pct columns (divide by 100) at consumption time, never in place.'""")

print("\nremaining district_unmatched (post-NFHS-5 districts by design):")
(gold.where("_district_match_status = 'district_unmatched'")
     .groupBy("district", "statename").count().orderBy(F.desc("count")).limit(10).show(truncate=False))


## 7. Converting to a Python script

Each section maps to one function in `src/clean.py` (this notebook now leads — port these cells next):

| Notebook cell | Script function | Notes |
|---|---|---|
| Bronze ingest | `ingest_bronze()` | idempotent snapshot writes |
| Bronze QA | `qa_report()` | print + optionally raise on grain violations |
| Dedupe facilities | `dedupe_facilities()` | collapse identical dup ids; merge complementary; flag conflicts |
| Silver 1/2/3 | `clean_facilities()`, `build_pincode_map()`, `clean_nfhs()` | pure transforms |
| District alias map | `build_alias_map()` | Layer A aliases + Layer B guarded fuzzy |
| Gold | `build_gold()` | joins + flags + state cross-check |
| Gold QA | `main()` tail | assertions on counts and match rates |

**TODO status (2026-11-21):**

* Done — Fuzzy/alias district matching: two layers implemented; 1,366 of 1,565 unmatched facilities recovered via Layer A (verified; fuzzy layer guards the tail); 199 facilities on post-NFHS-5 districts (NTR, Chengalpattu, Hanumakonda, Purba Bardhaman, …) intentionally left unmatched.
* Done — Country: hardcoded `India` (user-confirmed all facilities are India-based); `_country_backfilled` flags the rows whose declared value was junk/NULL.
* Done — NFHS casting: all 49 indicator string columns cast (`*` etc. → NULL), synced with `src/clean.py`.
* Done — State cross-check: `_state_disagrees` flag; pincode-derived state stays the join key.
* Done — Duplicate unique_ids: verified all 11 pairs are fully identical on non-audit columns; collapsed (conflicts would be kept + flagged `_dupe_conflict`).
* Done — pct scale: 0–100 canonical, declared in the gold table comment.
* Open decision: map post-NFHS-5 districts to their pre-split NFHS parents (NTR → Krishna?, Chengalpattu → Kancheepuram?) or leave unmatched.


In [0]:
%python
import runpy
mod = runpy.run_path(
    "/Workspace/Users/jdw2@andrew.cmu.edu/cmu_datathon26/src/catalog.py")
mod["main"](["--llm-min", "2"])

## 8. Platinum — token catalog & downstream chain (curation pass 2026-11-22)

The notebook leads through gold (§1–7). The platinum layer is built by the `src/` scripts, run via `runpy` — starting with the catalog cell above.

**Token curation:** the original LLM pass used `--llm-min 3`, so 4,334 tokens appearing in exactly 2 facilities were never classified. Re-running `catalog.py` with `--llm-min 2` (cell above) classified them once — answers cached forever in `platinum.channel_token_map_llm` (now 9,032 rows; future re-runs are free). Reviewing the new labels surfaced wrong or ambiguous mappings, now codified as deterministic rules in `catalog.py`:

* `RULES` additions: ACL/MPFL/knee-ligament → `arthroscopy` (LLM had put them in spine surgery); Velys/MAKO robots → `jointReplacementSurgery`; Q-switched laser → `dermatology` (was laserDentistry); Boyle's apparatus → `anesthesia` (was pulmonaryFunctionTesting); keratoplasty/PDEK → `corneaOphthalmology` (was missed entirely); pain clinic → physical medicine; gum depigmentation → `gumTreatment`.
* `NOT_SERVICE_PATTERNS` additions: vague claims the LLM force-mapped at conf < 0.5 ('revision surgery', 'advanced surgeries', 'regenerative therapies', 'laser machine', autoclaves, display monitors, 'state of the art', marketing/logistics text).
* Bug fix in `main()`: a cached LLM label can no longer override a curated not-service pattern (rules always win).

Result: repeated-token (n≥2) unmapped volume is **0**; every unmapped token is a singleton. After any token-map change, re-run the downstream chain so the new vocabulary flows into evidence, confidence bands and the QA report:

In [0]:
import runpy
mod = runpy.run_path(
    "/Workspace/Users/jdw2@andrew.cmu.edu/cmu_datathon26/src/evidence.py")
mod["main"]([])

In [0]:
import runpy
mod = runpy.run_path(
    "/Workspace/Users/jdw2@andrew.cmu.edu/cmu_datathon26/src/confidence.py")
mod["main"]([])

In [0]:
import runpy
mod = runpy.run_path(
    "/Workspace/Users/jdw2@andrew.cmu.edu/cmu_datathon26/src/report.py")
mod["main"]([])